<a href="https://colab.research.google.com/github/mahdisabetkish/Flickr-8k/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Download the data
!wget "https://github.com/awsaf49/flickr-dataset/releases/download/v1.0/flickr8k.zip"
!unzip -q flickr8k.zip -d ./flickr8k
!rm flickr8k.zip
!echo "Downloaded Flickr8k dataset successfully."

In [ ]:
import torch
import numpy as np
import albumentations as A
import os

from torch.utils.data import DataLoader, Dataset
from PIL import Image

class Flicker8KDataset(Dataset):
    def __init__(self, image_dir, caption_file):
        self.image_dir = image_dir
        self.caption_file = caption_file
        self.captions, self.ids = self.read_captions_id(caption_file)
        self.s2i = self.build_word_vocab()
        self.all_token = self.tokenizer()

        # Compute the maximum token length for the entire dataset
        self.max_token_len = self.compute_max_token_length()

    def compute_max_token_length(self):
        max_len = 0
        for image_id, captions in self.captions.items():
            for caption in captions:
                tokenized_caption = caption.strip().split()
                max_len = max(max_len, len(tokenized_caption))
        return max_len

    def build_word_vocab(self):
        words = set()
        captions = list(self.captions.values())
        for caption in range(len(captions)):
            for cap in range(len(captions[caption])):
                word = captions[caption][cap].strip().split(' ')
                words.update(word)
        s2i = {j: i for i, j in enumerate(words)}
        return s2i

    def tokenizer(self):
        all_token = {}
        for image_id, captions in self.captions.items():
            # Tokenize all captions for this image ID
            tokens = [[self.s2i[word] for word in caption.strip().split()] for caption in captions]
            all_token[image_id] = tokens  # Map tokens to the corresponding image ID
        return all_token

    def read_captions_id(self, caption_file):
        with open(caption_file, 'r') as f:
            lines = f.readlines()
            captions = {}
            ids = []
            for line in lines[1:]:
                image_id, caption = line.strip().split(',', 1)
                if image_id in captions:
                    captions[image_id].append(caption)
                else:
                    captions[image_id] = [caption]
                ids.append(image_id)
        return captions, ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, ix):
        image_path = os.path.join(self.image_dir, self.ids[ix])
        image = Image.open(image_path)
        image = np.array(image)
        transform = A.Resize(height=224, width=224)
        image = transform(image=image)['image']
        image = image.transpose((0, 1, 2))
        image = torch.tensor(image)

        cap = self.captions[self.ids[ix]]
        token = self.all_token[self.ids[ix]]
        return image, cap, token, self.ids[ix]

def Flicker8k_collate_fn(batch, max_token_len):
    """
    Custom collate function to handle variable-length captions and tokens.
    Pads captions and tokens to the same length (max_token_len) and aligns all
    samples to have the same number of captions.
    """
    images = []
    captions = []
    tokens = []
    ids = []

    max_caption_count = max(len(token) for _, _, token, _ in batch)  # Max captions per sample

    for image, caption, token, image_id in batch:
        images.append(image)
        captions.append(caption)

        # Pad captions within a sample
        padded_captions = [
            t + [0] * (max_token_len - len(t))  # Pad each caption
            for t in token
        ]

        # Add fully padded captions if the sample has fewer captions
        while len(padded_captions) < max_caption_count:
            padded_captions.append([0] * max_token_len)

        tokens.append(torch.tensor(padded_captions))  # Convert to tensor
        ids.append(image_id)

    # Stack images and tokens
    images = torch.stack(images)  # Stack images into a single tensor
    tokens = torch.stack(tokens)  # Stack tokens into a single tensor
    return images, captions, tokens, ids

# Initialize Dataset
image_dir = '/content/flickr8k/Images'
caption_file = '/content/flickr8k/captions.txt'

batch_size = 128
my_data = Flicker8KDataset(image_dir, caption_file)

# Use the max_token_len from the dataset instance
loader_flicker = DataLoader(my_data, batch_size=batch_size, shuffle=True,
                            collate_fn=lambda batch: Flicker8k_collate_fn(batch, my_data.max_token_len))

# Testing data loading
images, caption, token, id = next(iter(loader_flicker))

(type(token), len(token), len(token[1]), len(token[0][0]), token[1], token[1][1])
token.shape

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import albumentations as A
import os

# Load Pretrained VGG11 model and remove the classifier (fully connected layers)
vgg11 = models.vgg11(pretrained=True)
vgg11 = nn.Sequential(*list(vgg11.children())[:-1])

In [ ]:
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = 'cpu'
vgg11 = vgg11.to(device)

class Simplernn(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, target_length):
        super(Simplernn, self).__init__()
        self.sequence_projection = nn.Linear(49, target_length)  # Map 49 -> 38
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Project sequence length from 49 to target_length (38)
        x = x.permute(0, 2, 1)  # Shape: [batch_size, feature_dim, sequence_length]
        x = self.sequence_projection(x)  # Shape: [batch_size, feature_dim, target_length]
        x = x.permute(0, 2, 1)  # Shape: [batch_size, target_length, feature_dim]

        # Pass through RNN
        out, h = self.rnn(x)  # Shape: [batch_size, target_length, hidden_size]
        fc_out = self.fc(out)  # Shape: [batch_size, target_length, output_size]

        return fc_out, h


input_size = 512
hidden_size = 512
output_size = 9862 # it is the size of flicker8k vocabulary
target_length = 38

rnn_model = Simplernn(input_size, hidden_size, output_size, target_length)

learning_rate = 0.01
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=learning_rate)
epoch = 1
counter = 0

# Move the model to the chosen device (GPU or CPU)
rnn_model = rnn_model.to(device)
stepi = []; lossi = []
for i in range(epoch):
    rnn_model.train()
    for images, _, token, _ in loader_flicker:
        optimizer.zero_grad()
        images = images.permute(0, 3, 1, 2).to(device)
        token = token.to(device)  # Ensure token is on the correct device

        # Extract features from VGG11
        with torch.no_grad():
            features = vgg11(images.float())
            flatten_features = features.reshape(features.size(0), -1, 512)

        # Pass features through the RNN model
        out, hidden = rnn_model(flatten_features)  # Shape: [batch_size, target_length, output_size]

        # Ensure model output is float
        out = out.float()

        # Use only the first caption for loss calculation
        first_caption = token[:, 0, :]  # Shape: [batch_size, sequence_length] = [4, 38]

        # Compute the loss
        loss = criterion(out.contiguous().view(-1, output_size), first_caption.contiguous().view(-1))

        stepi.append(counter); lossi.append(loss.item()); counter += 1
        if counter % 100 == 0:
          print(f"Loss: {loss.item()}")

        # Backpropagation
        loss.backward()
        optimizer.step()

In [ ]:
import matplotlib.pyplot as lt

plt.plot(stepi, lossi)